In [ ]:
# NB_Log_Quarantine
# Called by pipeline when an unsupported file is detected
# Logs the quarantine event to quarantine_log Delta table

from pyspark.sql.types import StructType, StructField, StringType, TimestampType, BooleanType, LongType
import notebookutils
from datetime import datetime
from pyspark.sql import Row
import uuid

# Get parameters passed from pipeline
# params    = notebookutils.notebook.parameters
# file_name = params.get("file_name", "unknown")
# file_path = params.get("file_path", "unknown")

file_name = locals().get("file_name", "unknown")
file_path = locals().get("file_path", "unknown")
run_id    = locals().get("run_id", "manual")

# Determine file extension
ext = file_name.split(".")[-1] if "." in file_name else "unknown"

schema = StructType([
    StructField("quarantine_id", StringType(), True),
    StructField("run_id", StringType(), True),
    StructField("file_name", StringType(), True),
    StructField("file_path", StringType(), True),
    StructField("file_extension", StringType(), True),
    StructField("quarantine_reason", StringType(), True),
    StructField("quarantine_ts", TimestampType(), True),
    StructField("file_size_bytes", LongType(), True),
    StructField("resolved", BooleanType(), True),
    StructField("resolved_by", StringType(), True),
    StructField("resolved_ts", TimestampType(), True),
    StructField("notes", StringType(), True)
])

# 2. Create the row data
data = [(
    str(uuid.uuid4()),
    run_id,
    file_name,
    f"Files/quarantine/{file_name}",
    ext,
    f"Unsupported file format: .{ext}",
    datetime.utcnow(),
    0,
    False,
    None,
    None,
    "File moved to quarantine/ folder"
)]


# Write to quarantine_log
# import uuid
# spark.createDataFrame([Row(
#     quarantine_id     = str(uuid.uuid4()),
#     run_id            = run_id ,  #params.get("run_id", "manual"),
#     file_name         = file_name,
#     file_path         = f"Files/quarantine/{file_name}",
#     file_extension    = ext,
#     quarantine_reason = f"Unsupported file format: .{ext}",
#     quarantine_ts     = datetime.utcnow(),
#     file_size_bytes   = 0,
#     resolved          = False,
#     resolved_by       = None,
#     resolved_ts       = None,
#     notes             = "File moved to quarantine/ folder"
# )]).write.format("delta").mode("append") \
#   .saveAsTable("Blackwood_Bronze_Lakehouse.quarantine_log")

# 3. Create DataFrame and Write to quarantine_log
df = spark.createDataFrame(data, schema=schema)
df.write.format("delta").mode("append").saveAsTable("Blackwood_Bronze_Lakehouse.quarantine_log")

print(f"Quarantine logged: {file_name} (.{ext})")